In [0]:
import json
import os
from dotenv import load_dotenv
from confluent_kafka import Consumer, KafkaError

load_dotenv()

connection_string = os.getenv("CONNECTION_STR")
eventhub_name = os.getenv("EVENTHUB_NAME")
bootstrap_servers = os.getenv("BOOTSTRAP_SERVERS")

conf = {
    'bootstrap.servers': bootstrap_servers, 
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',  
    'sasl.password': connection_string,    
    'group.id': '$Default',  
    'auto.offset.reset': 'earliest'
}

consumer = Consumer(conf)
topic = eventhub_name 
consumer.subscribe([topic])

print("Listening for data")

try:
    while True:
        msg = consumer.poll(timeout=1.0)
        
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                print(f"Consumer error: {msg.error()}")
                break

        try:
            raw_data = msg.value().decode('utf-8')
            payload = json.loads(raw_data)
            
            if "data" in payload:
                payload = payload["data"]
            
            event_type = payload.get("e")
            symbol = payload.get("s")
            
            kline = payload.get("k", {})
            close_price = kline.get("c")
            open_price = kline.get("o")
            interval = kline.get("i")
            
            print(f"[{symbol} - {interval}] Open: {open_price} | Close: {close_price}")
            
        except json.JSONDecodeError:
            print("Failed to decode JSON. Raw message:", msg.value())

except KeyboardInterrupt:
    print("Stopping consumer...")
finally:
    consumer.close()